<a href="https://colab.research.google.com/github/Sourav1429/Machine_Unlearning/blob/main/iris_unlearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The working of Machine Unlearning by corrupting data and then making it forget

1) DTC

2) SVM

3) CNN

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
import cv2
import os
import random
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D,MaxPooling2D,Flatten,Dense

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
class type_of_data:
  def __init__(self,choice,path):
    self.c = choice
    self.p = path
  def return_data(self):
    X,y = [],[]
    if(self.c==1):
      data = pd.read_excel(self.p)
      X,y = data[data.columns[0:-1]].values,data[data.columns[-1]].values
    elif(self.c==2):
      labels = os.listdir(path)
      for i in range(len(labels)):
        complete_path = os.path.join(self.p,labels[i])
        for j in os.listdir(complete_path):
          img = cv2.imread(os.path.join(complete_path,j))
          img = cv2.resize(img,(128,128))
          X.append(img)
          y.append(i)
      X,y = np.array(X),np.array(y)
    return X,y

In [ ]:
class define_model:
  def __init__(self,model,X_train,y_train):
    self.m = model
    self.X = X_train
    self.y = y_train
  def train(self):
    if(self.m==0):
      model = DecisionTreeClassifier()
      model.fit(self.X,self.y)
    elif(self.m==1):
      model = SVC()
      model.fit(self.X,self.y)
    elif(self.m==2):
      self.X = self.X/255
      inp_shape = self.X.shape[1:]
      model = Sequential()
      model.add(Conv2D(32,(3,3),activation='relu',input_shape=inp_shape))
      model.add(MaxPooling2D((2,2)))
      model.add(Conv2D(64,(3,3),activation='relu'))
      model.add(MaxPooling2D((2,2)))
      model.add(Conv2D(64,(3,3),activation='relu'))
      model.add(Flatten())
      model.add(Dense(64,activation='relu'))
      model.add(Dense(10,activation='softmax'))
      model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])
      model.fit(self.X,self.y,epochs=10)
    return model

In [ ]:
path = "/content/drive/MyDrive/iris.xlsx"
choice = 1
data = type_of_data(choice,path)
X,y = data.return_data()
unique_labels = np.unique(y)
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=42)
model_choice = 1 # 0,1,2----- 0 - DTC, 1 - SVM , 2 - CNN
model_tr = define_model(model_choice,X_train,y_train)
model = model_tr.train()

After we have defined our models, we are going to test the models

1) Original accuracy

In [ ]:
if(model_choice==0):
  print(model.score(X_test,y_test))
elif(model_choice==1):
  print(model.score(X_test,y_test))
elif(model_choice==2):
  print(model.evaluate(X_test,y_test))
#model.predict(X_test)
#check prdiction of some data

1.0


2) Corrupt few labels and thn get the accuracy

In [ ]:
y_train_corr = y_train.copy()
ratio = 0.2 #ratio of total data points to be corrupted
size = int(ratio*len(X_train))
#print(y_train)
#print("===============")
corrupted_indices = np.array(random.sample(range(0, len(y_train)), size))
for i in corrupted_indices:
  y_train_corr[i] = np.random.choice(unique_labels)
  print(y_train[i],"======>",y_train_corr[i])
corr_model_tr = define_model(model_choice,X_train,y_train_corr)
corr_model = corr_model_tr.train()
if(model_choice==0):
  print(corr_model.score(X_test,y_test))
elif(model_choice==1):
  print(corr_model.score(X_test,y_test))
elif(model_choice==2):
  print(corr_model.evaluate(X_test,y_test))

2 ======> 0
1 ======> 1
2 ======> 0
2 ======> 1
2 ======> 2
2 ======> 1
1 ======> 0
0 ======> 2
2 ======> 0
1 ======> 1
2 ======> 1
2 ======> 1
2 ======> 2
2 ======> 1
0 ======> 0
1 ======> 1
0 ======> 2
2 ======> 1
0 ======> 0
1 ======> 2
2 ======> 0
0.9333333333333333


3) Perform Retraining by removing the corrupted data points and perform re-training

In [ ]:
#First step is deleting the not required indices
X_train_del = np.delete(X_train,corrupted_indices,axis=0)
y_train_del = np.delete(y_train,corrupted_indices,axis=0)
del_model_tr = define_model(model_choice,X_train_del,y_train_del)
del_model = del_model_tr.train()
if(model_choice==0):
  print(del_model.score(X_test,y_test))
elif(model_choice==1):
  print(del_model.score(X_test,y_test))
elif(model_choice==2):
  print(del_model.evaluate(X_test,y_test))

0.9777777777777777


4) Perform unlearning and then obtain accuracy

In [ ]:
#Creating an augmented dataset
X_train_aug,y_train_aug = [],[]
for i in range(len(corrupted_indices)):
  for j in unique_labels:
    if(j!=y_train[corrupted_indices[i]]):
      X_train_aug.append(X_train[corrupted_indices[i]])
      y_train_aug.append(j)
X_train_aug,y_train_aug = np.array(X_train_aug),np.array(y_train_aug)
X_concat = np.concatenate((X_train,X_train_aug),axis=0)
y_concat = np.concatenate((y_train,y_train_aug),axis=0)
print(X_concat.shape)
print(y_concat.shape)

(147, 4)
(147,)


In [ ]:
aug_model_tr = define_model(model_choice,X_concat,y_concat)
aug_model = aug_model_tr.train()
if(model_choice==0):
  print(aug_model.score(X_test,y_test))
elif(model_choice==1):
  print(aug_model.score(X_test,y_test))
elif(model_choice==2):
  print(aug_model.evaluate(X_test,y_test))

0.9777777777777777


2) DTC

In [ ]:
path = "/content/drive/MyDrive/iris.xlsx"
choice = 1
data = type_of_data(choice,path)
X,y = data.return_data()
unique_labels = np.unique(y)
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=12)
model_choice = 0 # 0,1,2----- 0 - DTC, 1 - SVM , 2 - CNN
model_tr = define_model(model_choice,X_train,y_train)
model = model_tr.train()
print("Original model accuracy:")
if(model_choice==0):
  print(model.score(X_test,y_test))
elif(model_choice==1):
  print(model.score(X_test,y_test))
elif(model_choice==2):
  print(model.evaluate(X_test,y_test))

print("Corrupted model accuracy:")
y_train_corr = y_train.copy()
ratio = 0.2 #ratio of total data points to be corrupted
size = int(ratio*len(X_train))
#print(y_train)
#print("===============")
corrupted_indices = np.array(random.sample(range(0, len(y_train)), size))
for i in corrupted_indices:
  y_train_corr[i] = np.random.choice(unique_labels)
  print(y_train[i],"======>",y_train_corr[i])
corr_model_tr = define_model(model_choice,X_train,y_train_corr)
corr_model = corr_model_tr.train()
if(model_choice==0):
  print(corr_model.score(X_test,y_test))
elif(model_choice==1):
  print(corr_model.score(X_test,y_test))
elif(model_choice==2):
  print(corr_model.evaluate(X_test,y_test))

print("Retraining accuracy")
del_model_tr = define_model(model_choice,X_train_del,y_train_del)
del_model = del_model_tr.train()
if(model_choice==0):
  print(del_model.score(X_test,y_test))
elif(model_choice==1):
  print(del_model.score(X_test,y_test))
elif(model_choice==2):
  print(del_model.evaluate(X_test,y_test))


print("Unlearning accuracy")
aug_model_tr = define_model(model_choice,X_concat,y_concat)
aug_model = aug_model_tr.train()
if(model_choice==0):
  print(aug_model.score(X_test,y_test))
elif(model_choice==1):
  print(aug_model.score(X_test,y_test))
elif(model_choice==2):
  print(aug_model.evaluate(X_test,y_test))

Original model accuracy:
0.9333333333333333
Corrupted model accuracy:
2 ======> 2
1 ======> 0
1 ======> 1
1 ======> 0
2 ======> 1
2 ======> 0
2 ======> 1
2 ======> 0
1 ======> 1
2 ======> 1
1 ======> 2
2 ======> 2
0 ======> 2
1 ======> 2
2 ======> 1
1 ======> 0
0 ======> 2
0 ======> 1
1 ======> 0
1 ======> 1
1 ======> 1
1 ======> 2
1 ======> 1
2 ======> 2
0.6333333333333333
Retraining accuracy
1.0
Unlearning accuracy
0.8333333333333334
